# Linear Regression - Comprehensive Examples

This notebook provides step-by-step examples of linear regression techniques.

## Table of Contents
1. [Simple Linear Regression](#1.-Simple-Linear-Regression)
2. [Multiple Linear Regression](#2.-Multiple-Linear-Regression)
3. [Polynomial Regression](#3.-Polynomial-Regression)
4. [Regularization (Ridge, Lasso, Elastic Net)](#4.-Regularization)
5. [Model Diagnostics](#5.-Model-Diagnostics)
6. [Real-World Example](#6.-Real-World-Example)

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Simple Linear Regression

### Theory Recap
Simple linear regression models the relationship between one independent variable (X) and one dependent variable (Y):

$$Y = \beta_0 + \beta_1 X + \epsilon$$

### Step-by-Step Example

In [ ]:
# STEP 1: Load and explore the data
df_simple = pd.read_csv('../datasets/linear_regression_simple.csv')

print("Dataset shape:", df_simple.shape)
print("\nFirst few rows:")
print(df_simple.head())
print("\nBasic statistics:")
print(df_simple.describe())

In [ ]:
# STEP 2: Visualize the relationship
plt.figure(figsize=(10, 6))
plt.scatter(df_simple['X'], df_simple['y'], alpha=0.6, edgecolors='k')
plt.xlabel('X (Independent Variable)', fontsize=12)
plt.ylabel('y (Dependent Variable)', fontsize=12)
plt.title('Scatter Plot: Exploring the Relationship', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

# Calculate correlation
correlation = df_simple['X'].corr(df_simple['y'])
print(f"Correlation coefficient: {correlation:.4f}")

In [ ]:
# STEP 3: Prepare data
X = df_simple[['X']].values  # Must be 2D array
y = df_simple['y'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# STEP 4: Create and train the model
model_simple = LinearRegression()
model_simple.fit(X, y)

# Extract parameters
intercept = model_simple.intercept_
slope = model_simple.coef_[0]

print(f"Intercept (β₀): {intercept:.4f}")
print(f"Slope (β₁): {slope:.4f}")
print(f"\nEquation: y = {intercept:.4f} + {slope:.4f} * X")

In [ ]:
# STEP 5: Make predictions
y_pred = model_simple.predict(X)

# Show first few predictions
comparison = pd.DataFrame({
    'X': X.ravel(),
    'Actual': y,
    'Predicted': y_pred,
    'Residual': y - y_pred
})
print(comparison.head(10))

In [ ]:
# STEP 6: Evaluate the model
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)

print("Model Performance Metrics:")
print(f"R² Score: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

print(f"\nInterpretation:")
print(f"  - The model explains {r2*100:.2f}% of the variance in y")
print(f"  - Average prediction error (RMSE): {rmse:.4f}")

In [ ]:
# STEP 7: Visualize the results
plt.figure(figsize=(12, 5))

# Plot 1: Regression line
plt.subplot(1, 2, 1)
plt.scatter(X, y, alpha=0.6, label='Actual data', edgecolors='k')
plt.plot(X, y_pred, color='red', linewidth=2, label=f'Regression line\ny = {intercept:.2f} + {slope:.2f}X')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Simple Linear Regression Fit', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Residuals
plt.subplot(1, 2, 2)
residuals = y - y_pred
plt.scatter(y_pred, residuals, alpha=0.6, edgecolors='k')
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values', fontsize=12)
plt.ylabel('Residuals', fontsize=12)
plt.title('Residual Plot', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Manual Implementation (Understanding the Math)

In [ ]:
# Calculate slope and intercept manually
x_mean = X.mean()
y_mean = y.mean()

# β₁ = Cov(X, Y) / Var(X)
numerator = np.sum((X.ravel() - x_mean) * (y - y_mean))
denominator = np.sum((X.ravel() - x_mean) ** 2)
slope_manual = numerator / denominator

# β₀ = ȳ - β₁x̄
intercept_manual = y_mean - slope_manual * x_mean

print("Manual Calculation:")
print(f"Slope (β₁): {slope_manual:.4f}")
print(f"Intercept (β₀): {intercept_manual:.4f}")

print("\nScikit-learn:")
print(f"Slope (β₁): {slope:.4f}")
print(f"Intercept (β₀): {intercept:.4f}")

print("\n✓ Results match!")

## 2. Multiple Linear Regression

### Theory Recap
Multiple linear regression extends simple regression to multiple predictors:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_p X_p + \epsilon$$

### Step-by-Step Example

In [ ]:
# STEP 1: Load and explore the data
df_multi = pd.read_csv('../datasets/linear_regression_multiple.csv')

print("Dataset shape:", df_multi.shape)
print("\nFirst few rows:")
print(df_multi.head())
print("\nColumn names:")
print(df_multi.columns.tolist())

In [ ]:
# STEP 2: Analyze correlations
plt.figure(figsize=(10, 8))
correlation_matrix = df_multi.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Correlations with target
target_corr = correlation_matrix['target'].sort_values(ascending=False)
print("\nCorrelations with target:")
print(target_corr)

In [ ]:
# STEP 3: Prepare data
feature_cols = [col for col in df_multi.columns if col != 'target']
X_multi = df_multi[feature_cols].values
y_multi = df_multi['target'].values

print(f"Features: {feature_cols}")
print(f"X shape: {X_multi.shape}")
print(f"y shape: {y_multi.shape}")

In [ ]:
# STEP 4: Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# STEP 5: Train the model
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)

# Display coefficients
print("Model Coefficients:")
print(f"Intercept: {model_multi.intercept_:.4f}")
print("\nFeature Coefficients:")
for feature, coef in zip(feature_cols, model_multi.coef_):
    print(f"  {feature}: {coef:.4f}")

In [ ]:
# STEP 6: Make predictions
y_train_pred = model_multi.predict(X_train)
y_test_pred = model_multi.predict(X_test)

# STEP 7: Evaluate performance
print("Training Set Performance:")
print(f"  R²: {r2_score(y_train, y_train_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")

print("\nTest Set Performance:")
print(f"  R²: {r2_score(y_test, y_test_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")

# Check for overfitting
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
if train_r2 - test_r2 > 0.1:
    print("\n⚠️ Warning: Possible overfitting detected!")
else:
    print("\n✓ Model generalizes well to test data")

In [ ]:
# STEP 8: Visualize predictions vs actual
plt.figure(figsize=(12, 5))

# Training set
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.6, edgecolors='k')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Values', fontsize=12)
plt.ylabel('Predicted Values', fontsize=12)
plt.title(f'Training Set (R² = {train_r2:.4f})', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Test set
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.6, edgecolors='k', color='orange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Values', fontsize=12)
plt.ylabel('Predicted Values', fontsize=12)
plt.title(f'Test Set (R² = {test_r2:.4f})', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 9: Feature importance (based on absolute coefficients)
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model_multi.coef_,
    'Abs_Coefficient': np.abs(model_multi.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("\nFeature Importance (by absolute coefficient):")
print(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'])
plt.xlabel('Coefficient Value', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Coefficients', fontsize=14)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Polynomial Regression

### Theory Recap
Polynomial regression fits a non-linear relationship by creating polynomial features:

$$Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + ... + \epsilon$$

It's still a linear model (linear in coefficients), just with transformed features.

In [ ]:
# STEP 1: Load polynomial data
df_poly = pd.read_csv('../datasets/polynomial_regression.csv')

X_poly = df_poly[['X']].values
y_poly = df_poly['y'].values

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X_poly, y_poly, alpha=0.6, edgecolors='k')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Non-linear Relationship', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print("Notice: The relationship is clearly non-linear (quadratic)")

In [ ]:
# STEP 2: Try linear regression first (to show it's inadequate)
model_linear = LinearRegression()
model_linear.fit(X_poly, y_poly)
y_linear_pred = model_linear.predict(X_poly)

r2_linear = r2_score(y_poly, y_linear_pred)
print(f"Linear Model R²: {r2_linear:.4f}")
print("This is poor! Let's try polynomial features...")

In [ ]:
# STEP 3: Create polynomial features and fit models of different degrees
degrees = [1, 2, 3, 5, 10]
plt.figure(figsize=(15, 10))

for idx, degree in enumerate(degrees, 1):
    # Create polynomial features
    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_transformed = poly_features.fit_transform(X_poly)
    
    # Fit model
    model_poly = LinearRegression()
    model_poly.fit(X_poly_transformed, y_poly)
    
    # Predictions
    y_poly_pred = model_poly.predict(X_poly_transformed)
    r2 = r2_score(y_poly, y_poly_pred)
    
    # Plot
    plt.subplot(2, 3, idx)
    plt.scatter(X_poly, y_poly, alpha=0.5, edgecolors='k', label='Data')
    
    # Sort for smooth line
    sort_idx = X_poly.ravel().argsort()
    plt.plot(X_poly[sort_idx], y_poly_pred[sort_idx], 
             color='red', linewidth=2, label=f'Degree {degree} (R²={r2:.4f})')
    
    plt.xlabel('X', fontsize=11)
    plt.ylabel('y', fontsize=11)
    plt.title(f'Polynomial Degree {degree}', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("  - Degree 1: Underfits (too simple)")
print("  - Degree 2: Good fit (matches the true quadratic relationship)")
print("  - Degree 10: Overfits (too complex, wiggly line)")

## 4. Regularization

### Theory Recap
Regularization prevents overfitting by adding a penalty to the loss function:

- **Ridge (L2)**: Penalizes sum of squared coefficients
- **Lasso (L1)**: Penalizes sum of absolute coefficients (can produce zero coefficients)
- **Elastic Net**: Combination of L1 and L2

In [ ]:
# Create high-degree polynomial features (prone to overfitting)
poly_features_high = PolynomialFeatures(degree=10, include_bias=False)
X_poly_high = poly_features_high.fit_transform(X_poly)

# Split data
X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly_high, y_poly, test_size=0.2, random_state=42
)

# Standardize features (important for regularization!)
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

print(f"Number of features after polynomial transformation: {X_poly_high.shape[1]}")
print("Features are standardized for fair regularization")

In [ ]:
# Compare different models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=0.1)': Ridge(alpha=0.1),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Ridge (α=10)': Ridge(alpha=10.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'Elastic Net': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

results = []

for name, model in models.items():
    # Train
    model.fit(X_train_poly_scaled, y_train_poly)
    
    # Predict
    y_train_pred = model.predict(X_train_poly_scaled)
    y_test_pred = model.predict(X_test_poly_scaled)
    
    # Evaluate
    train_r2 = r2_score(y_train_poly, y_train_pred)
    test_r2 = r2_score(y_test_poly, y_test_pred)
    
    # Count non-zero coefficients
    if hasattr(model, 'coef_'):
        non_zero = np.sum(np.abs(model.coef_) > 1e-5)
    else:
        non_zero = X_train_poly_scaled.shape[1]
    
    results.append({
        'Model': name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Overfitting': train_r2 - test_r2,
        'Non-zero Coefs': non_zero
    })

results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

print("\nKey Insights:")
print("  - Linear Regression: Likely overfits (high train R², low test R²)")
print("  - Ridge: Reduces overfitting, all coefficients non-zero")
print("  - Lasso: Feature selection (some coefficients become exactly zero)")
print("  - Higher α = more regularization = simpler model")

In [ ]:
# Visualize coefficient shrinkage
alphas = np.logspace(-3, 3, 50)
ridge_coefs = []
lasso_coefs = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_poly_scaled, y_train_poly)
    ridge_coefs.append(ridge.coef_)
    
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_poly_scaled, y_train_poly)
    lasso_coefs.append(lasso.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for i in range(ridge_coefs.shape[1]):
    plt.plot(alphas, ridge_coefs[:, i], alpha=0.6)
plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)', fontsize=12)
plt.ylabel('Coefficient Value', fontsize=12)
plt.title('Ridge: Coefficient Paths', fontsize=14)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for i in range(lasso_coefs.shape[1]):
    plt.plot(alphas, lasso_coefs[:, i], alpha=0.6)
plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)', fontsize=12)
plt.ylabel('Coefficient Value', fontsize=12)
plt.title('Lasso: Coefficient Paths (Note: Some go to zero!)', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice:")
print("  - Ridge: Coefficients shrink but never reach exactly zero")
print("  - Lasso: Coefficients can become exactly zero (sparse model)")

## 5. Model Diagnostics

Checking assumptions and model quality

In [ ]:
# Use the simple linear regression model for diagnostics
residuals = y - model_simple.predict(X)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted (check linearity and homoscedasticity)
axes[0, 0].scatter(y_pred, residuals, alpha=0.6, edgecolors='k')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values', fontsize=11)
axes[0, 0].set_ylabel('Residuals', fontsize=11)
axes[0, 0].set_title('Residuals vs Fitted\n(Check: Random scatter around 0)', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q Plot (check normality of residuals)
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot\n(Check: Points follow diagonal line)', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# 3. Scale-Location (check homoscedasticity)
standardized_residuals = residuals / residuals.std()
axes[1, 0].scatter(y_pred, np.sqrt(np.abs(standardized_residuals)), alpha=0.6, edgecolors='k')
axes[1, 0].set_xlabel('Fitted Values', fontsize=11)
axes[1, 0].set_ylabel('√|Standardized Residuals|', fontsize=11)
axes[1, 0].set_title('Scale-Location\n(Check: Horizontal line)', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

# 4. Histogram of Residuals (check normality)
axes[1, 1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Residuals', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Histogram of Residuals\n(Check: Approximately normal)', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Diagnostic Checks:")
print("  1. Residuals vs Fitted: Should show random scatter (no patterns)")
print("  2. Q-Q Plot: Points should roughly follow the diagonal line")
print("  3. Scale-Location: Should show horizontal band (constant variance)")
print("  4. Histogram: Should look approximately bell-shaped (normal)")

In [ ]:
# Statistical tests
from scipy.stats import shapiro, normaltest

print("Statistical Tests:")
print("\n1. Shapiro-Wilk Test (Normality of Residuals):")
stat, p_value = shapiro(residuals)
print(f"   Test Statistic: {stat:.4f}")
print(f"   P-value: {p_value:.4f}")
if p_value > 0.05:
    print("   ✓ Residuals appear normally distributed (p > 0.05)")
else:
    print("   ✗ Residuals may not be normally distributed (p < 0.05)")

print("\n2. Mean of Residuals (should be ≈ 0):")
print(f"   Mean: {residuals.mean():.6f}")
print("   ✓ Very close to zero!")

## 6. Real-World Example: Diabetes Dataset

Complete workflow from data to deployment-ready model

In [ ]:
# STEP 1: Load data
df_diabetes = pd.read_csv('../datasets/diabetes.csv')

print("Dataset Information:")
print(f"Shape: {df_diabetes.shape}")
print(f"\nColumns: {df_diabetes.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_diabetes.head())

In [ ]:
# STEP 2: EDA
print("Basic Statistics:")
print(df_diabetes.describe())

# Check for missing values
print(f"\nMissing Values:")
print(df_diabetes.isnull().sum())

In [ ]:
# STEP 3: Prepare data
X_diabetes = df_diabetes.drop('target', axis=1).values
y_diabetes = df_diabetes['target'].values

# Split
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

# Standardize
scaler_d = StandardScaler()
X_train_d_scaled = scaler_d.fit_transform(X_train_d)
X_test_d_scaled = scaler_d.transform(X_test_d)

print(f"Training set: {X_train_d.shape}")
print(f"Test set: {X_test_d.shape}")

In [ ]:
# STEP 4: Train multiple models and compare
diabetes_models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Ridge (α=10)': Ridge(alpha=10.0),
    'Lasso (α=1)': Lasso(alpha=1.0),
    'Elastic Net': ElasticNet(alpha=1.0, l1_ratio=0.5)
}

diabetes_results = []

for name, model in diabetes_models.items():
    # Train
    model.fit(X_train_d_scaled, y_train_d)
    
    # Predict
    y_train_pred_d = model.predict(X_train_d_scaled)
    y_test_pred_d = model.predict(X_test_d_scaled)
    
    # Evaluate
    train_r2 = r2_score(y_train_d, y_train_pred_d)
    test_r2 = r2_score(y_test_d, y_test_pred_d)
    test_rmse = np.sqrt(mean_squared_error(y_test_d, y_test_pred_d))
    test_mae = mean_absolute_error(y_test_d, y_test_pred_d)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_d_scaled, y_train_d, 
                                 cv=5, scoring='r2')
    
    diabetes_results.append({
        'Model': name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'CV R² (mean)': cv_scores.mean(),
        'CV R² (std)': cv_scores.std(),
        'Test RMSE': test_rmse,
        'Test MAE': test_mae
    })

diabetes_results_df = pd.DataFrame(diabetes_results)
print("\nModel Comparison on Diabetes Dataset:")
print(diabetes_results_df.to_string(index=False))

# Select best model
best_model_idx = diabetes_results_df['Test R²'].idxmax()
best_model_name = diabetes_results_df.loc[best_model_idx, 'Model']
print(f"\n🏆 Best Model: {best_model_name}")

## Summary

In this notebook, we covered:

1. **Simple Linear Regression**: One predictor, interpretable relationship
2. **Multiple Linear Regression**: Multiple predictors, feature importance
3. **Polynomial Regression**: Handling non-linear relationships
4. **Regularization**: Ridge, Lasso, and Elastic Net for preventing overfitting
5. **Model Diagnostics**: Checking assumptions and model quality
6. **Real-World Application**: Complete workflow on diabetes dataset

### Key Takeaways:

- Always visualize your data first
- Check model assumptions
- Use train-test split to evaluate generalization
- Consider regularization for high-dimensional data
- Compare multiple models before selecting the best one
- Use cross-validation for robust evaluation

### Next Steps:

- Review `theory.md` for deeper mathematical understanding
- Check `INTERVIEW_PREP.md` for quick reference and common interview questions
- Try these techniques on your own datasets!